# 02. 좌표 변환 연습

**목표:** 로봇의 좌표계 사이를 행렬로 오가는 법을 직접 구현합니다.
이 단계가 끝나면 "카메라가 본 점", "로봇 베이스", "펜 끝" 사이의 좌표를 계산할 수 있습니다.

**진행 방법**
1. 각 문제의 `raise NotImplementedError` 를 지우고 직접 구현하세요.
2. 바로 아래 채점 셀을 실행하면 ✅ / ❌ 와 힌트가 나옵니다.
3. 막히면 먼저 힌트를 읽고, 그래도 안 되면 `solution.py` 를 참고하세요. (바로 보면 연습 효과가 떨어집니다.)

**약속(규약)**
- `T_a_b` 는 **b 좌표계에서 표현된 점을 a 좌표계로 옮기는** 4x4 행렬입니다. (b 좌표계의 원점이 a 기준 어디에 있고 어떻게 돌아 있는지)
- 각도는 라디안, 길이는 미터.
- 쿼터니언은 `(w, x, y, z)` 순서입니다. (MuJoCo와 같음. ROS 메시지는 `x, y, z, w` 순서라서 섞이기 쉽습니다!)

In [ ]:
import numpy as np
from checker import *
np.set_printoptions(precision=4, suppress=True)

## 문제 1. 기본 회전행렬

x, y, z 축을 중심으로 `a` 라디안 회전하는 3x3 행렬을 만드세요.

- 오른손 법칙: 엄지를 회전축 방향으로 향했을 때 나머지 손가락이 감기는 방향이 +입니다.
- 힌트: z축 회전은 `[[cos, -sin, 0], [sin, cos, 0], [0, 0, 1]]` 입니다. x, y축은 어떤 행/열이 "고정"되는지 생각해 보세요.

In [ ]:
def rot_x(a):
    raise NotImplementedError

def rot_y(a):
    raise NotImplementedError

def rot_z(a):
    raise NotImplementedError

check_rotations(rot_x, rot_y, rot_z)

### 생각해 볼 문제 (코드로 확인해 보세요)
회전은 **순서가 중요**합니다. 아래 셀에서 `Rx(90°) @ Rz(90°)` 와 `Rz(90°) @ Rx(90°)` 가 같은지 확인하고, 다르다면 왜 다른지 한 줄로 적어보세요.

In [ ]:
# 여기서 직접 실험해 보세요 (정답 체크 없음)

## 문제 2. 동차변환행렬 (4x4)

회전 `R` (3x3) 과 이동 `p` (3,) 를 하나의 4x4 행렬로 묶습니다.

```
T = [ R  p ]
    [ 0  1 ]
```

구현할 것:
- `make_T(R, p)` : 4x4 행렬 만들기
- `invert_T(T)` : 역변환. **`np.linalg.inv` 를 쓰지 말고** `R.T` 를 이용하세요. (왜 R의 역행렬이 전치일까요? `R.T @ R = I` 를 떠올려 보세요.) 이동 성분은 `-R.T @ p` 가 됩니다.
- `transform_point(T, p)` : 점 `p` 를 변환. 점에는 끝에 `1`을 붙여서(동차좌표) `T` 와 곱합니다.
- `compose(*Ts)` : 여러 변환을 **왼쪽부터** 차례로 곱하기

시나리오: 로봇 베이스에서 (1, 0, 0.5) 위치에 카메라가 달려 있고, 카메라는 z축으로 90° 돌아 있습니다. 카메라가 `(1, 0, 0)` 에서 물체를 봤다면 베이스 기준 좌표는? 손으로 먼저 계산해 보고 채점으로 확인하세요.

In [ ]:
def make_T(R, p):
    raise NotImplementedError

def invert_T(T):
    raise NotImplementedError

def transform_point(T, p):
    raise NotImplementedError

def compose(*Ts):
    raise NotImplementedError

check_transforms(make_T, invert_T, transform_point, compose)

### 개념 퀴즈 1
`T_a_b` 는 b의 점을 a로 옮기고, `T_b_c` 는 c의 점을 b로 옮깁니다. c의 점을 a로 옮기는 행렬은?

- `A`: `T_a_b @ T_b_c`
- `B`: `T_b_c @ T_a_b`

(힌트: 점은 **오른쪽**에서 곱해집니다. 가장 먼저 적용되는 변환이 가장 오른쪽에 있어야 합니다.)

In [ ]:
quiz1 = None   # "A" 또는 "B"
check_quiz("퀴즈 1", quiz1, "A", "점 p_c 에 먼저 적용되는 변환은 T_b_c 입니다. 점은 오른쪽에 있습니다.")

## 문제 3. 오일러각 (roll, pitch, yaw) ↔ 회전행렬

로봇 문서와 티치펜던트는 자세를 `(roll, pitch, yaw)` 로 보여주는 경우가 많습니다.
이 노트북의 규약은 **`R = Rz(yaw) @ Ry(pitch) @ Rx(roll)`** 입니다. (ROS의 RPY와 같음. 한화 컨트롤러의 규약은 다를 수 있으니 실제 로봇을 쓸 때는 꼭 확인하세요!)

구현할 것:
- `rpy_to_R(roll, pitch, yaw)`
- `R_to_rpy(R)` : 반환 범위는 `pitch ∈ [-90°, 90°]`.
  - `R[2, 0] = -sin(pitch)`, `R[2, 1] = cos(pitch)·sin(roll)`, `R[2, 2] = cos(pitch)·cos(roll)`, `R[1, 0] = sin(yaw)·cos(pitch)`, `R[0, 0] = cos(yaw)·cos(pitch)` 를 이용하세요.
  - **짐벌락:** `pitch = ±90°` 면 `cos(pitch) = 0` 이라서 `atan2(0, 0)` 이 됩니다. 이때는 roll 과 yaw 를 따로 구할 수 없으니 `yaw = 0` 으로 고정하고 roll 에 합치세요.
    - `pitch = +90°` 일 때 `R[0,1] = sin(roll - yaw)`, `R[1,1] = cos(roll - yaw)`
    - `pitch = -90°` 일 때 `R[0,1] = -sin(roll + yaw)`, `R[1,1] = cos(roll + yaw)`

In [ ]:
def rpy_to_R(roll, pitch, yaw):
    raise NotImplementedError

def R_to_rpy(R):
    raise NotImplementedError

check_euler(rpy_to_R, R_to_rpy, rot_x, rot_y, rot_z)

### 생각해 볼 문제
짐벌락에 가까운 자세(예: pitch = 89.9°)에서 로봇의 roll/yaw 표시값이 갑자기 크게 튀는 이유를 위 설명을 바탕으로 설명해 보세요. 왜 로봇 내부 계산에는 오일러각보다 회전행렬이나 쿼터니언을 쓸까요?

## 문제 4. 축-각 표현 (로드리게스 공식)

"단위 벡터 `axis` 를 축으로 `angle` 만큼 회전"을 3x3 회전행렬로 바꿉니다.

```
R = I + sin(θ)·K + (1 - cos(θ))·K²      (K 는 axis 의 반대칭행렬)
K = [[ 0, -z,  y],
     [ z,  0, -x],
     [-y,  x,  0]]
```

- 축이 단위 벡터가 아닐 수도 있으니 먼저 정규화하세요.

In [ ]:
def axis_angle_to_R(axis, angle):
    raise NotImplementedError

check_axis_angle(axis_angle_to_R, rot_z)

## 문제 5. 쿼터니언 ↔ 회전행렬

ROS의 자세 메시지, MuJoCo, 많은 로봇 SDK가 쿼터니언을 씁니다. `(w, x, y, z)` 순서, 단위 길이, `w ≥ 0` 으로 통일합니다.
(`q` 와 `-q` 는 같은 회전이라 부호를 하나로 정해야 비교가 쉽습니다.)

- `quat_to_R(q)` : 아래 공식을 그대로 옮기면 됩니다. (`q` 는 먼저 정규화)
  ```
  R = [[1-2(y²+z²), 2(xy-zw),   2(xz+yw)  ],
       [2(xy+zw),   1-2(x²+z²), 2(yz-xw)  ],
       [2(xz-yw),   2(yz+xw),   1-2(x²+y²)]]
  ```
- `R_to_quat(R)` : 어렵습니다. **`trace(R) > 0` 이면** `w = sqrt(1 + trace)/2` 로 구하고 `x = (R[2,1]-R[1,2])/(4w)` 같은 식을 씁니다. **`trace(R) ≤ 0` 이면** 대각 성분 중 가장 큰 것을 기준으로 다른 분기를 써야 합니다. (예: x축 180° 회전은 `trace = -1`.) 막히면 `scipy.spatial.transform.Rotation` 이나 `solution.py` 와 비교해 보세요.

In [ ]:
def R_to_quat(R):
    raise NotImplementedError

def quat_to_R(q):
    raise NotImplementedError

check_quaternion(R_to_quat, quat_to_R, rpy_to_R)

## 문제 6. 실제 HCR-3A 에서 확인하기

이 저장소의 MuJoCo 모델(`hcr3a_pick_place/scene.xml`)로 좌표계를 곱해서 TCP 위치를 재현합니다.

- 월드 → `link_6` : MuJoCo 가 계산해 준 값을 `make_T` 로 감쌉니다.
- `link_6` → `tcp` : 모델에 적힌 상수 변환입니다 (`link_6` 끝에서 0.214 m 앞).
- 이 둘을 `compose` 로 곱한 결과가 MuJoCo 가 직접 계산한 TCP 위치와 **같아야** 합니다.

곱하는 순서를 바꾸면 얼마나 틀어지는지도 직접 확인해 보세요. (채점 메시지에 mm 단위 오차가 나옵니다.)

In [ ]:
check_with_mujoco(compose, make_T)

## 마무리 점검

위에서 ✅ 가 아닌 항목이 있다면 다시 확인하세요. 모두 통과했다면 다음 질문에 말로 답해 보세요. (면접에서 자주 나옵니다.)

1. 회전행렬의 역행렬이 전치인 이유는?
2. `T_a_b` 의 역행렬의 이동 성분이 `-p` 가 아니라 `-R.T @ p` 인 이유는?
3. 오일러각의 짐벌락은 무엇이고, 왜 쿼터니언은 이 문제가 없나?
4. 카메라가 본 점을 로봇 베이스 좌표로 바꾸려면 어떤 행렬이 필요한가? (→ 이후 **핸드아이 캘리브레이션** 단계에서 이 행렬을 구합니다)

다음 단계: **03. 순기구학** — 관절각에서 TCP 의 `T` 를 계산합니다.